# UBS Colab Worker Node (Browser)\nThis notebook is the **Worker Node**. It runs the browser and connects out to Cloud Run signaling server.

In [ ]:
!pip install -q playwright flask requests pyngrok\n!playwright install chromium

In [ ]:
import os, requests, threading, time, uuid\nfrom flask import Flask, request, jsonify\nfrom playwright.sync_api import sync_playwright\nfrom pyngrok import ngrok\n\nSIGNAL_URL = os.getenv('UBS_SIGNAL_URL', 'https://YOUR-CLOUD-RUN-URL')\nREGION = os.getenv('UBS_REGION', 'us')\nNODE_ID = os.getenv('UBS_NODE_ID', 'colab-' + str(uuid.uuid4())[:8])

In [ ]:
app = Flask(__name__)\n\n@app.route('/run', methods=['POST'])\ndef run():\n    data = request.get_json() or {}\n    url = data.get('url')\n    with sync_playwright() as p:\n        browser = p.chromium.launch(headless=True)\n        page = browser.new_page()\n        page.goto(url, wait_until='domcontentloaded')\n        title = page.title()\n        html = page.content()[:5000]\n        browser.close()\n    return jsonify({'ok': True, 'title': title, 'htmlSnippet': html})\n\nthreading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

In [ ]:
public_url = ngrok.connect(5000, bind_tls=True).public_url\nprint('Worker connect URL:', public_url)\n\nregister_payload = {\n  'id': NODE_ID,\n  'connectUrl': public_url,\n  'region': REGION,\n  'capabilities': ['playwright']\n}\nr = requests.post(f'{SIGNAL_URL}/signal/register', json=register_payload, timeout=20)\nprint('Register status:', r.status_code, r.text)

In [ ]:
try:\n  while True:\n    hb = requests.post(f'{SIGNAL_URL}/signal/heartbeat', json={'id': NODE_ID}, timeout=20)\n    print('heartbeat', hb.status_code)\n    time.sleep(240)\nexcept KeyboardInterrupt:\n  print('Stopping worker node heartbeat')

## ✅ SUCCESS\nWorker node is live and connected to signaling server. You can close this tab.